In [ ]:
import os
import pandas as pd
import zipfile
import openpyxl
from win32com.client import Dispatch
import time

In [ ]:
pasta_mes = os.getenv('PATH_FOLDER')
pasta_principal = os.path.dirname(os.path.dirname(pasta_mes))
mes = pasta_mes.split(" - ")[1][0:3]

mes

In [ ]:
def compacta_pastas(caminho_principal, mes):
    # Itera sobre cada pasta dentro do caminho principal
    for cliente in os.listdir(caminho_principal):
        caminho_cliente = os.path.join(caminho_principal, cliente)
        lista_todos_clientes.append(cliente)

        # Verifica se é uma pasta
        if os.path.isdir(caminho_cliente):
            # Nome do arquivo zip
            nome_zip = f'KIT {mes} 25 - {cliente}.zip'
            caminho_zip = os.path.join(caminho_cliente, nome_zip)
            
            # Cria o arquivo zip
            with zipfile.ZipFile(caminho_zip, 'w') as zipf:
                # Percorre recursivamente todos os arquivos e subpastas
                for root, _, files in os.walk(caminho_cliente):
                    for file in files:
                        arquivo_path = os.path.join(root, file)
                        # Ignora o próprio arquivo zipado recém-criado
                        if arquivo_path == caminho_zip:
                            continue
                        # Adiciona ao zip mantendo a estrutura relativa
                        zipf.write(arquivo_path, os.path.relpath(arquivo_path, caminho_cliente))
            
            print(f"Arquivos de {cliente} compactados em: {caminho_zip}")


In [ ]:
lista_todos_clientes = []

compacta_pastas(pasta_mes, mes)

In [ ]:
caminho_lista_emails = os.path.join(pasta_principal,'lista_emails_corrigido.xlsx')

df = pd.read_excel(caminho_lista_emails)

df

In [ ]:
novos_clientes = [c for c in lista_todos_clientes if c not in df['CLIENTE'].values]

novos_clientes

In [ ]:
df = pd.concat([df, pd.DataFrame({"CLIENTE": novos_clientes})], ignore_index=True)

df

In [ ]:
df.to_excel(caminho_lista_emails, index=False)

In [ ]:
df = pd.read_excel(caminho_lista_emails)

df

In [ ]:
import os
import logging
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.application import MIMEApplication
from dotenv import load_dotenv

# Carrega variáveis de ambiente
load_dotenv()


def send_email(sender_email, to_email, subject, zip_path, cliente):

    # Monta o e-mail
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = ", ".join(to_email)
    msg['Subject'] = subject
    cc_emails = [
        'cs@tyrenergia.com.br',
        'servicosfinanceiros@tyrenergia.com.br',
    ]

    msg['Cc'] = ", ".join(cc_emails)

    body = """Olá, tudo bem?<br><br>

Segue anexo pasta zipada, contendo os pagamentos feitos pela Tyr referente aos encargos de energia elétrica, atendido no Mercado Livre de Energia.<br>
Estamos implementando a automação no processo de envio dos kits, e em breve você receberá os documentos de forma totalmente automatizada. Nesta etapa, já iniciamos parte desse processo.<br><br>

Caso precise de mais informações ou desejar entrar em contato, por favor utilize o seguinte email: cs@tyrenergia.com.br <br><br>

Att.,<br>
Serviços Financeiros<br>
<a href="https://tyrenergia.com.br/">
  <img src="https://conexaomercadolivre.com.br/wp-content/uploads/2023/10/tyr-01-2.png" alt="Logo" width="170">
</a><br>

<a href="https://www.instagram.com/tyr_energia/">
  <img src="https://imgur.com/8ORzh3H.png" alt="Logo" width="30">
</a>     
<a href="https://www.linkedin.com/company/tyr-energia/">
  <img src="https://imgur.com/aLXboB1.png" alt="Logo" width="30">
</a>
<a href="https://www.facebook.com/tyrenergia">
  <img src="https://imgur.com/9HSuHaJ.png" alt="Logo" width="30">
</a><br>

<img src="https://imgur.com/Aj2JhZi.png" alt="Logo" width="250">

"""

    # Corpo do e-mail
    msg.attach(MIMEText(body, 'html'))

    # Anexo ZIP
    with open(zip_path, 'rb') as f:
        part = MIMEApplication(f.read(), Name=os.path.basename(zip_path))
        part['Content-Disposition'] = f'attachment; filename="{os.path.basename(zip_path)}"'
        msg.attach(part)

    try:
          # Inclui os destinatários + cópia na lista de envio
          all_recipients = to_email + cc_emails
          server.sendmail(sender_email, all_recipients, msg.as_string())
          print(f'E-mail enviado para {cliente} com email: {to_email}')

    except Exception as e:
        print(f'Falha no envio para {to_email}: {e}')


In [ ]:
smtp_server = "smtp.office365.com"
smtp_port = 587
sender_email = os.getenv('SENDER_EMAIL')
login_email = os.getenv('LOGIN_EMAIL')
login_password = os.getenv('LOGIN_PASSWORD')

df_emails_enviados = pd.DataFrame(columns=['CLIENTE'])

with smtplib.SMTP(smtp_server, smtp_port) as server:
    server.starttls()
    server.login(login_email, login_password)
    
    for _, row in df.iterrows():  # Itera sobre cada linha
        cliente = row['CLIENTE']
        email_cliente = [e.strip() for e in row['EMAIL'].split(';')]

        caminho_cliente = os.path.join(pasta_mes, cliente)
        
        # Envia o arquivo compactado para o cliente
        nome_zip = f'KIT {mes} 25 - {cliente}.zip'
        caminho_zip = os.path.join(caminho_cliente, nome_zip)
        
        if os.path.exists(caminho_zip):
            assunto = f'KIT - PRESTAÇÃO DE CONTAS | {cliente} | {mes} 25'
            send_email(sender_email, email_cliente, assunto, caminho_zip, cliente)

            df_emails_enviados.loc[len(df_emails_enviados)] = {'CLIENTE': cliente}
            
        else:
            print(f'O arquivo {nome_zip} não foi encontrado para {cliente}.')

In [ ]:
df_emails_enviados.to_excel(os.path.join(pasta_principal,'emails_enviados.xlsx'))